# SQL Coding Reference Notebook
**Written:** 07/21/2026

This notebook is a reference guide for SQL **(assisted by ClaudeAI for time efficiency)**, organized into three sections:

1. **General** — standard ANSI SQL that works the same (or nearly the same) in both MySQL and PostgreSQL
2. **MySQL** — syntax and features unique to MySQL
3. **PostgreSQL** — syntax and features unique to PostgreSQL

Each topic includes:
- A short explanation of what the code does
- A sample query snippet
- A hand-written sample output showing what the result would look like

> **Note:** The queries here are illustrative. They are not executed against a live database. The 'output' cells show hand-crafted example results so you can see the shape of the result set without needing a DB connection.

## 1. General SQL

Syntax and concepts that are part of the ANSI SQL standard and behave the same way in both MySQL and PostgreSQL.

### 1.1 SELECT & WHERE

Retrieves specific columns from a table, filtered by a condition. `SELECT` chooses which columns to return, and `WHERE` filters rows before they're returned.

In [1]:
query = """
SELECT employee_id, first_name, department, salary
FROM employees
WHERE department = 'Engineering'
  AND salary > 60000;
"""
print(query)


SELECT employee_id, first_name, department, salary
FROM employees
WHERE department = 'Engineering'
  AND salary > 60000;



In [2]:
print("""employee_id | first_name | department  | salary
-------------+------------+-------------+--------
     101     |   Maria    | Engineering | 72000
     104     |   James    | Engineering | 65500
     109     |   Aiko     | Engineering | 81000""")

employee_id | first_name | department  | salary
-------------+------------+-------------+--------
     101     |   Maria    | Engineering | 72000
     104     |   James    | Engineering | 65500
     109     |   Aiko     | Engineering | 81000


### 1.2 JOIN (INNER & LEFT)

Combines rows from two or more tables based on a related column. `INNER JOIN` returns only matching rows; `LEFT JOIN` returns all rows from the left table, with NULLs where there's no match on the right.

In [3]:
query = """
SELECT e.first_name, e.department, m.project_name
FROM employees e
LEFT JOIN projects m
    ON e.employee_id = m.lead_id;
"""
print(query)


SELECT e.first_name, e.department, m.project_name
FROM employees e
LEFT JOIN projects m
    ON e.employee_id = m.lead_id;



In [4]:
print("""first_name | department  |  project_name
------------+-------------+-----------------
   Maria    | Engineering | Platform Rewrite
   James    | Engineering | NULL
   Aiko     | Engineering | Data Pipeline""")

first_name | department  |  project_name
------------+-------------+-----------------
   Maria    | Engineering | Platform Rewrite
   James    | Engineering | NULL
   Aiko     | Engineering | Data Pipeline


### 1.3 GROUP BY & Aggregate Functions

Groups rows sharing the same value in specified columns so aggregate functions (COUNT, SUM, AVG, MIN, MAX) can be applied to each group.

In [5]:
query = """
SELECT department, COUNT(*) AS headcount, AVG(salary) AS avg_salary
FROM employees
GROUP BY department
ORDER BY avg_salary DESC;
"""
print(query)


SELECT department, COUNT(*) AS headcount, AVG(salary) AS avg_salary
FROM employees
GROUP BY department
ORDER BY avg_salary DESC;



In [6]:
print("""department  | headcount | avg_salary
-------------+-----------+------------
 Engineering |     3     |  72833.33
 Sales       |     2     |  58250.00
 HR          |     1     |  51000.00""")

department  | headcount | avg_salary
-------------+-----------+------------
 Engineering |     3     |  72833.33
 Sales       |     2     |  58250.00
 HR          |     1     |  51000.00


### 1.4 ORDER BY & LIMIT

`ORDER BY` sorts the result set (ASC by default, or DESC). `LIMIT` restricts how many rows are returned — both MySQL and PostgreSQL support the basic `LIMIT n` form.

In [7]:
query = """
SELECT first_name, salary
FROM employees
ORDER BY salary DESC
LIMIT 3;
"""
print(query)


SELECT first_name, salary
FROM employees
ORDER BY salary DESC
LIMIT 3;



In [8]:
print("""first_name | salary
------------+--------
   Aiko     | 81000
   Maria    | 72000
   James    | 65500""")

first_name | salary
------------+--------
   Aiko     | 81000
   Maria    | 72000
   James    | 65500


### 1.5 Subqueries

A query nested inside another query, often used in the WHERE clause to filter based on the result of another query.

In [9]:
query = """
SELECT first_name, salary
FROM employees
WHERE salary > (
    SELECT AVG(salary) FROM employees
);
"""
print(query)


SELECT first_name, salary
FROM employees
WHERE salary > (
    SELECT AVG(salary) FROM employees
);



In [10]:
print("""first_name | salary
------------+--------
   Maria    | 72000
   Aiko     | 81000""")

first_name | salary
------------+--------
   Maria    | 72000
   Aiko     | 81000


### 1.6 Common Table Expressions (WITH)

A CTE creates a named temporary result set that can be referenced later in the query. Improves readability compared to nested subqueries, and both engines support it.

In [11]:
query = """
WITH dept_avg AS (
    SELECT department, AVG(salary) AS avg_salary
    FROM employees
    GROUP BY department
)
SELECT e.first_name, e.department, d.avg_salary
FROM employees e
JOIN dept_avg d ON e.department = d.department
WHERE e.salary > d.avg_salary;
"""
print(query)


WITH dept_avg AS (
    SELECT department, AVG(salary) AS avg_salary
    FROM employees
    GROUP BY department
)
SELECT e.first_name, e.department, d.avg_salary
FROM employees e
JOIN dept_avg d ON e.department = d.department
WHERE e.salary > d.avg_salary;



In [12]:
print("""first_name | department  | avg_salary
------------+-------------+------------
   Maria    | Engineering |  72833.33
   Aiko     | Engineering |  72833.33""")

first_name | department  | avg_salary
------------+-------------+------------
   Maria    | Engineering |  72833.33
   Aiko     | Engineering |  72833.33


### 1.7 CASE WHEN

Adds conditional (if/else-style) logic directly inside a SQL statement, producing a derived column based on one or more conditions.

In [13]:
query = """
SELECT first_name, salary,
    CASE
        WHEN salary >= 70000 THEN 'Senior'
        WHEN salary >= 55000 THEN 'Mid'
        ELSE 'Junior'
    END AS pay_band
FROM employees;
"""
print(query)


SELECT first_name, salary,
    CASE
        WHEN salary >= 70000 THEN 'Senior'
        WHEN salary >= 55000 THEN 'Mid'
        ELSE 'Junior'
    END AS pay_band
FROM employees;



In [14]:
print("""first_name | salary | pay_band
------------+--------+----------
   Maria    | 72000  |  Senior
   James    | 65500  |   Mid
   Aiko     | 81000  |  Senior""")

first_name | salary | pay_band
------------+--------+----------
   Maria    | 72000  |  Senior
   James    | 65500  |   Mid
   Aiko     | 81000  |  Senior


## 2. MySQL

Syntax and features that are unique to (or notably different in) MySQL.

### 2.1 Backtick Identifiers

MySQL uses backticks (`` ` ``) to quote identifiers (table/column names), which is handy when a name clashes with a reserved word or contains spaces. PostgreSQL uses double quotes for this instead.

In [15]:
query = """
SELECT `order`, `date`
FROM `sales_2024`
WHERE `order` > 1000;
"""
print(query)


SELECT `order`, `date`
FROM `sales_2024`
WHERE `order` > 1000;



In [16]:
print("""order | date
-------+------------
 1250  | 2024-03-02
 1980  | 2024-03-04""")

order | date
-------+------------
 1250  | 2024-03-02
 1980  | 2024-03-04


### 2.2 AUTO_INCREMENT

MySQL's way of auto-generating sequential primary key values on insert. PostgreSQL achieves the same thing with SERIAL or GENERATED ALWAYS AS IDENTITY.

In [17]:
query = """
CREATE TABLE customers (
    customer_id INT AUTO_INCREMENT PRIMARY KEY,
    name VARCHAR(100),
    signup_date DATE
);

INSERT INTO customers (name, signup_date)
VALUES ('Diego Reyes', '2024-05-01');
"""
print(query)


CREATE TABLE customers (
    customer_id INT AUTO_INCREMENT PRIMARY KEY,
    name VARCHAR(100),
    signup_date DATE
);

INSERT INTO customers (name, signup_date)
VALUES ('Diego Reyes', '2024-05-01');



In [18]:
print("""customer_id |    name     | signup_date
-------------+-------------+-------------
      1      | Diego Reyes | 2024-05-01""")

customer_id |    name     | signup_date
-------------+-------------+-------------
      1      | Diego Reyes | 2024-05-01


### 2.3 IFNULL()

Returns a fallback value if the given expression is NULL. PostgreSQL uses the ANSI-standard COALESCE() for this (MySQL also supports COALESCE, but IFNULL is its own shorthand).

In [19]:
query = """
SELECT name, IFNULL(phone, 'Not Provided') AS phone
FROM customers;
"""
print(query)


SELECT name, IFNULL(phone, 'Not Provided') AS phone
FROM customers;



In [20]:
print("""name     |    phone
--------------+---------------
 Diego Reyes  | Not Provided
 Lin Zhou     | 555-0192""")

name     |    phone
--------------+---------------
 Diego Reyes  | Not Provided
 Lin Zhou     | 555-0192


### 2.4 GROUP_CONCAT()

Concatenates values from a group into a single string, with an optional separator. PostgreSQL's equivalent is STRING_AGG().

In [21]:
query = """
SELECT department, GROUP_CONCAT(first_name SEPARATOR ', ') AS team_members
FROM employees
GROUP BY department;
"""
print(query)


SELECT department, GROUP_CONCAT(first_name SEPARATOR ', ') AS team_members
FROM employees
GROUP BY department;



In [22]:
print("""department  |      team_members
-------------+---------------------------
 Engineering | Maria, James, Aiko
 Sales       | Wendy, Tomas""")

department  |      team_members
-------------+---------------------------
 Engineering | Maria, James, Aiko
 Sales       | Wendy, Tomas


### 2.5 ON DUPLICATE KEY UPDATE

MySQL's 'upsert' syntax: inserts a row, but if it violates a unique/primary key constraint, updates the existing row instead. PostgreSQL's equivalent is INSERT ... ON CONFLICT DO UPDATE.

In [23]:
query = """
INSERT INTO inventory (sku, quantity)
VALUES ('SKU-100', 50)
ON DUPLICATE KEY UPDATE quantity = quantity + 50;
"""
print(query)


INSERT INTO inventory (sku, quantity)
VALUES ('SKU-100', 50)
ON DUPLICATE KEY UPDATE quantity = quantity + 50;



In [24]:
print("""Before:  sku=SKU-100, quantity=120
After:   sku=SKU-100, quantity=170   (50 added to existing stock)""")

Before:  sku=SKU-100, quantity=120
After:   sku=SKU-100, quantity=170   (50 added to existing stock)


### 2.6 STR_TO_DATE() / DATE_FORMAT()

MySQL-specific functions for parsing strings into dates and formatting dates back into strings, using printf-style format codes (e.g. %Y, %m, %d).

In [25]:
query = """
SELECT
    STR_TO_DATE('03-15-2024', '%m-%d-%Y') AS parsed_date,
    DATE_FORMAT(NOW(), '%W, %M %e') AS pretty_today;
"""
print(query)


SELECT
    STR_TO_DATE('03-15-2024', '%m-%d-%Y') AS parsed_date,
    DATE_FORMAT(NOW(), '%W, %M %e') AS pretty_today;



In [26]:
print("""parsed_date | pretty_today
-------------+---------------------------
 2024-03-15  | Sunday, July 21""")

parsed_date | pretty_today
-------------+---------------------------
 2024-03-15  | Sunday, July 21


### 2.7 LIMIT with Offset Shorthand

MySQL supports a comma-shorthand `LIMIT offset, count` in addition to the standard `LIMIT count OFFSET offset` — a syntax PostgreSQL does not support.

In [27]:
query = """
-- Skip the first 5 rows, then return the next 3
SELECT first_name, salary
FROM employees
ORDER BY salary DESC
LIMIT 5, 3;
"""
print(query)


-- Skip the first 5 rows, then return the next 3
SELECT first_name, salary
FROM employees
ORDER BY salary DESC
LIMIT 5, 3;



In [28]:
print("""first_name | salary
------------+--------
   Wendy    | 54000
   Tomas    | 52500
   Nora     | 49900""")

first_name | salary
------------+--------
   Wendy    | 54000
   Tomas    | 52500
   Nora     | 49900


## 3. PostgreSQL

Syntax and features that are unique to (or notably different in) PostgreSQL.

### 3.1 Double-Quoted Identifiers & SERIAL

PostgreSQL uses double quotes to quote identifiers (case-sensitive names or reserved words), and uses SERIAL (or GENERATED ALWAYS AS IDENTITY) instead of MySQL's AUTO_INCREMENT.

In [29]:
query = """
CREATE TABLE "customers" (
    customer_id SERIAL PRIMARY KEY,
    "name" VARCHAR(100),
    signup_date DATE
);

INSERT INTO "customers" ("name", signup_date)
VALUES ('Diego Reyes', '2024-05-01');
"""
print(query)


CREATE TABLE "customers" (
    customer_id SERIAL PRIMARY KEY,
    "name" VARCHAR(100),
    signup_date DATE
);

INSERT INTO "customers" ("name", signup_date)
VALUES ('Diego Reyes', '2024-05-01');



In [30]:
print("""customer_id |    name     | signup_date
-------------+-------------+-------------
      1      | Diego Reyes | 2024-05-01""")

customer_id |    name     | signup_date
-------------+-------------+-------------
      1      | Diego Reyes | 2024-05-01


### 3.2 STRING_AGG()

PostgreSQL's version of concatenating grouped values into a single delimited string (MySQL's equivalent is GROUP_CONCAT).

In [31]:
query = """
SELECT department, STRING_AGG(first_name, ', ') AS team_members
FROM employees
GROUP BY department;
"""
print(query)


SELECT department, STRING_AGG(first_name, ', ') AS team_members
FROM employees
GROUP BY department;



In [32]:
print("""department  |      team_members
-------------+---------------------------
 Engineering | Maria, James, Aiko
 Sales       | Wendy, Tomas""")

department  |      team_members
-------------+---------------------------
 Engineering | Maria, James, Aiko
 Sales       | Wendy, Tomas


### 3.3 INSERT ... ON CONFLICT ... RETURNING

PostgreSQL's 'upsert' syntax (ON CONFLICT DO UPDATE), combined with RETURNING to get back the affected row(s) in the same statement — MySQL has no direct RETURNING equivalent.

In [33]:
query = """
INSERT INTO inventory (sku, quantity)
VALUES ('SKU-100', 50)
ON CONFLICT (sku)
DO UPDATE SET quantity = inventory.quantity + 50
RETURNING sku, quantity;
"""
print(query)


INSERT INTO inventory (sku, quantity)
VALUES ('SKU-100', 50)
ON CONFLICT (sku)
DO UPDATE SET quantity = inventory.quantity + 50
RETURNING sku, quantity;



In [34]:
print("""sku   | quantity
---------+----------
 SKU-100 |   170""")

sku   | quantity
---------+----------
 SKU-100 |   170


### 3.4 Arrays & unnest()

PostgreSQL has a native array type, letting a single column store a list of values. unnest() expands an array into one row per element. MySQL has no native array type.

In [35]:
query = """
SELECT name, tags
FROM articles
WHERE 'sql' = ANY(tags);

-- Expand the tags array into individual rows
SELECT name, unnest(tags) AS tag
FROM articles;
"""
print(query)


SELECT name, tags
FROM articles
WHERE 'sql' = ANY(tags);

-- Expand the tags array into individual rows
SELECT name, unnest(tags) AS tag
FROM articles;



In [36]:
print("""-- ANY() filter
    name           |          tags
--------------------+---------------------------
 Intro to Postgres  | {sql, database, postgres}

-- unnest() expansion
    name           |   tag
--------------------+-----------
 Intro to Postgres  | sql
 Intro to Postgres  | database
 Intro to Postgres  | postgres""")

-- ANY() filter
    name           |          tags
--------------------+---------------------------
 Intro to Postgres  | {sql, database, postgres}

-- unnest() expansion
    name           |   tag
--------------------+-----------
 Intro to Postgres  | sql
 Intro to Postgres  | database
 Intro to Postgres  | postgres


### 3.5 ILIKE (Case-Insensitive Matching)

PostgreSQL's ILIKE operator performs a case-insensitive pattern match, as opposed to LIKE which is case-sensitive by default in Postgres (MySQL's LIKE is case-insensitive by default on most collations, so it has no separate ILIKE).

In [37]:
query = """
SELECT first_name, department
FROM employees
WHERE department ILIKE 'engineer%';
"""
print(query)


SELECT first_name, department
FROM employees
WHERE department ILIKE 'engineer%';



In [38]:
print("""first_name | department
------------+-------------
   Maria    | Engineering
   James    | Engineering
   Aiko     | Engineering""")

first_name | department
------------+-------------
   Maria    | Engineering
   James    | Engineering
   Aiko     | Engineering


### 3.6 generate_series()

A PostgreSQL set-returning function that produces a sequence of values (numbers, dates, etc.) — frequently used to build calendar tables or fill gaps in reporting data. MySQL has no built-in equivalent.

In [39]:
query = """
SELECT generate_series(
    '2024-01-01'::date,
    '2024-01-05'::date,
    interval '1 day'
) AS report_date;
"""
print(query)


SELECT generate_series(
    '2024-01-01'::date,
    '2024-01-05'::date,
    interval '1 day'
) AS report_date;



In [40]:
print("""report_date
-------------
 2024-01-01
 2024-01-02
 2024-01-03
 2024-01-04
 2024-01-05""")

report_date
-------------
 2024-01-01
 2024-01-02
 2024-01-03
 2024-01-04
 2024-01-05
